In [1]:
# %% [markdown]
# # Textile Defect Detection (Kaggle TextileDefectDetection) - Clean Baseline
# This notebook keeps the original pipeline and paths, but is reorganized for clarity.
#
# Pipeline:
# 1) Merge raw train/test CSV + H5 into a single processed dataset
# 2) Compute MD5 hashes to analyze exact duplicates / leakage
# 3) Deduplicate within each original split, then stratified Train/Val split
# 4) Train a small CNN classifier with Early Stopping


In [2]:
import copy
import hashlib
import json
import math
import os
import random
from collections import defaultdict
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import tqdm
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    import torchinfo
except ImportError:
    torchinfo = None

try:
    import torch_directml  # type: ignore
except ImportError:
    torch_directml = None

try:
    from captum.attr import IntegratedGradients
    from captum.attr import visualization as viz
except ImportError:
    IntegratedGradients = None
    viz = None


In [3]:
# %% [markdown]
# ## 1. Imports & Configuration

# Disable HDF5 file locking for better compatibility
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Path configuration (DO NOT CHANGE)
ROOT = Path.cwd()
RAW = ROOT / "data" / "raw" / "textile"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

# File paths (DO NOT CHANGE)
TRAIN_H5, TRAIN_CSV = RAW / "train64.h5", RAW / "train64.csv"
TEST_H5, TEST_CSV = RAW / "test64.h5", RAW / "test64.csv"
OUT_H5, OUT_CSV = PROCESSED / "full64.h5", PROCESSED / "full64.csv"


def select_device():
    '''
    Priority:
    1) NVIDIA CUDA
    2) DirectML (Windows AMD/Intel/NVIDIA)
    3) Apple MPS
    4) CPU
    '''
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        return torch.device("cuda"), f"cuda ({gpu_name})"

    if torch_directml is not None:
        try:
            dml_device = torch_directml.device()
            return dml_device, "directml (AMD/Intel/NVIDIA)"
        except Exception as e:
            print(f"[WARN] DirectML detected but unavailable: {e}")

    if hasattr(torch, "backends") and hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps"), "mps (Apple Silicon)"

    return torch.device("cpu"), "cpu"


device, device_name = select_device()
print(f"Using device: {device_name}")
if (not torch.cuda.is_available()) and (torch_directml is None):
    print("[INFO] No CUDA/DirectML backend found. Running on CPU.")
    print("[INFO] For AMD on Windows, install DirectML: pip install torch-directml")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)


def _require_file(path: Path) -> None:
    '''Fail fast when a required file is missing.'''
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


def _normalize_label(x) -> str:
    return str(x).strip()


def print_class_counts(df: pd.DataFrame, title: str) -> None:
    '''Print total rows and per-class distribution for the given dataframe.'''
    if "indication_type" not in df.columns:
        print(f"[{title}] Missing column: indication_type")
        return

    vc = df["indication_type"].astype(str).str.strip().value_counts()
    print(f"\n[{title}] total_images={len(df)}")
    for k, v in vc.items():
        print(f"  {k}: {v}")


Using device: directml (AMD/Intel/NVIDIA)


In [ ]:
# %% [markdown]
# ## 1.5 Global Training Configuration
# Shared settings for all models (Baseline CNN + ResNet + future models)

FULL_CLASSES = ["good", "color", "cut", "hole", "thread", "metal_contamination"]

TRAIN_CFG = {
    "batch": 64,
    "lr": 0.001,
    "epochs": 30,
    "patience": 7,
}

BASELINE_SCENARIO = "all_training"


In [ ]:
# %% [markdown]
# ## 1.6 Global Split Scenarios (Reusable)
# One shared split table used by all models.

SPLIT_SCENARIOS = {
    # 1) Use all training data
    "all_training": {
        "defect_classes": FULL_CLASSES,
        "train_size": 47843,
        "defect_frac": 0.0,
        "desc": "Use all available deduplicated training data",
    },
    # 2) 50/50: 50% good + 50% defects (defect classes balanced)
    "fifty_fifty": {
        "defect_classes": FULL_CLASSES,
        "train_size": 20000,
        "defect_frac": 0.10,
        "desc": "50% good + 50% defects (evenly distributed across defect classes)",
    },
    # 3) Excluding two classes
    "exclude_two_classes": {
        "defect_classes": ["good", "color", "cut", "hole"],
        "train_size": 20000,
        "defect_frac": 0.0,
        "desc": "Train without two classes: thread, metal_contamination",
    },
    # 4) Imbalanced dataset
    "imbalanced": {
        "defect_classes": FULL_CLASSES,
        "train_size": 20000,
        "defect_frac": 0.02,
        "desc": "Strong class imbalance toward good samples",
    },
}

# Baseline CNN uses scenario #1.
BASELINE_SPLIT_CFG = SPLIT_SCENARIOS[BASELINE_SCENARIO]

for name, split_cfg in SPLIT_SCENARIOS.items():
    print(f"{name}: {split_cfg['desc']}")


In [4]:
# %% [markdown]
# ## 2. Merge Raw Train/Test into Processed Full Dataset

def merge_data() -> None:
    """
    Merge separate train/test H5 and CSV files into a unified dataset.

    Outputs:
      - data/processed/full64.csv
      - data/processed/full64.h5
    """
    if OUT_H5.exists() and OUT_CSV.exists():
        print("Dataset already merged. Skipping merge.")
        return

    _require_file(TRAIN_CSV)
    _require_file(TEST_CSV)
    _require_file(TRAIN_H5)
    _require_file(TEST_H5)

    df_train = pd.read_csv(TRAIN_CSV)
    df_test = pd.read_csv(TEST_CSV)

    # Keep original split info
    df_train["original_split"] = "train"
    df_test["original_split"] = "test"

    full_df = pd.concat([df_train, df_test], ignore_index=True)
    full_df.to_csv(OUT_CSV, index=False)
    print(f"Saved merged CSV: {OUT_CSV}")

    with h5py.File(OUT_H5, "w") as f_out:
        with h5py.File(TRAIN_H5, "r") as f_tr, h5py.File(TEST_H5, "r") as f_te:
            tr_imgs = f_tr["images"]
            te_imgs = f_te["images"]

            total_shape = (tr_imgs.shape[0] + te_imgs.shape[0], *tr_imgs.shape[1:])
            dset = f_out.create_dataset("images", shape=total_shape, dtype="f")  # keep original dtype choice

            dset[: tr_imgs.shape[0]] = tr_imgs[:]
            dset[tr_imgs.shape[0] :] = te_imgs[:]

    print(f"Saved merged H5: {OUT_H5}")


In [5]:
# %% [markdown]
# ## 3. MD5 Hashing & Duplicate Analysis

def get_h5_hashes(h5_path: Path, total_images: int, chunk_size: int = 5000) -> List[str]:
    """Generate MD5 fingerprints for all images in the H5 file."""
    hashes: List[str] = [""] * total_images
    print(f"Generating MD5 fingerprints for {total_images} images...")

    with h5py.File(h5_path, "r") as f:
        images = f["images"]
        for start in range(0, total_images, chunk_size):
            end = min(start + chunk_size, total_images)
            chunk = images[start:end]
            for i, img in enumerate(chunk):
                hashes[start + i] = hashlib.md5(img.tobytes()).hexdigest()

    return hashes


def analyze_duplicates() -> List[str]:
    """
    Identify exact duplicates via MD5 and check leakage across original splits.

    Output:
      - data/processed/duplicates_report.csv (if duplicates exist)
    """
    _require_file(OUT_CSV)
    _require_file(OUT_H5)

    df = pd.read_csv(OUT_CSV)
    with h5py.File(OUT_H5, "r") as f:
        total = int(f["images"].shape[0])

    all_hashes = get_h5_hashes(OUT_H5, total)

    hash_map: Dict[str, List[int]] = defaultdict(list)
    for idx, h in enumerate(all_hashes):
        hash_map[h].append(idx)

    dup_groups = {h: idxs for h, idxs in hash_map.items() if len(idxs) > 1}
    dup_rows = sum(len(idxs) for idxs in dup_groups.values())

    print(f"Duplicate groups={len(dup_groups)} | duplicate_rows={dup_rows}")

    if dup_groups:
        dup_indices = [i for idxs in dup_groups.values() for i in idxs]
        report_df = df.iloc[dup_indices].copy()
        report_df["md5"] = [all_hashes[i] for i in dup_indices]
        report_path = PROCESSED / "duplicates_report.csv"
        report_df.to_csv(report_path, index=False)
        print(f"Saved duplicates report: {report_path}")

        # Leakage check: same md5 appears in both original train and original test
        leakage = report_df.groupby("md5")["original_split"].nunique()
        if (leakage > 1).any():
            print("[WARNING] Data leakage detected across original splits (train/test)!")
        else:
            print("[SAFE] No leakage found among duplicates across original splits.")

    return all_hashes


In [6]:
# %% [markdown]
# ## 4. Split Generation (Dedup per original split + Stratified Train/Val)

def create_clean_split(all_hashes: List[str], included_classes: List[str], train_size: int, defect_frac: float) -> None:
    """
    Remove internal duplicates within each original split and generate Train/Val/Test CSVs.
    Remove requested classes from training set before splitting into final train and validation sets.

    Outputs:
      - data/processed/train_split.csv
      - data/processed/val_split.csv
      - data/processed/test_split.csv
    """
    df = pd.read_csv(OUT_CSV).copy()
    df["abs_ptr"] = range(len(df))  # pointer into full64.h5
    df["md5"] = all_hashes
    df["indication_type"] = df["indication_type"].astype(str).str.strip()

    tr_df_raw = df[df["original_split"] == "train"].copy()
    te_df_raw = df[df["original_split"] == "test"].copy()

    # Deduplicate within each portion
    tr_before, te_before = len(tr_df_raw), len(te_df_raw)
    tr_df = tr_df_raw.drop_duplicates(subset="md5", keep="first")
    te_df = te_df_raw.drop_duplicates(subset="md5", keep="first")
    tr_removed, te_removed = tr_before - len(tr_df), te_before - len(te_df)
    total_removed = tr_removed + te_removed

    print(f"Duplicates removed (within split): train={tr_removed}, test={te_removed}, total={total_removed}")

    # Keep only requested classes in training dataframe
    tr_df = tr_df[tr_df["indication_type"].isin(included_classes)].copy()

    # Sample desired fraction profile
    reduced_tr_df = reduce_training_set(tr_df, included_classes, train_size, defect_frac)
    reduced_tr_df = reduced_tr_df.sample(frac=1, random_state=42).reset_index(drop=True)

    split_col = "index" if "index" in reduced_tr_df.columns else "abs_ptr"

    # Stratified split (Train -> Train/Val) on unique sample key
    unique_df = reduced_tr_df.drop_duplicates(split_col)[[split_col, "indication_type"]].copy()
    label_counts = unique_df["indication_type"].value_counts()
    use_stratify = (label_counts.min() >= 2) and (label_counts.shape[0] > 1)
    stratify_labels = unique_df["indication_type"] if use_stratify else None
    if not use_stratify:
        print("[WARN] Stratified split disabled (insufficient per-class samples).")

    train_idx, val_idx = train_test_split(
        unique_df[split_col],
        test_size=0.1,
        random_state=42,
        stratify=stratify_labels,
    )

    df_train = reduced_tr_df[reduced_tr_df[split_col].isin(train_idx)].sample(frac=1, random_state=42)
    df_val = reduced_tr_df[reduced_tr_df[split_col].isin(val_idx)].sample(frac=1, random_state=42)

    train_path = PROCESSED / "train_split.csv"
    val_path = PROCESSED / "val_split.csv"
    test_path = PROCESSED / "test_split.csv"

    df_train.to_csv(train_path, index=False)
    df_val.to_csv(val_path, index=False)
    te_df.to_csv(test_path, index=False)

    print(f"Datasets finalized: Train({len(df_train)}), Val({len(df_val)}), Test({len(te_df)})")

    # Requested reporting
    print_class_counts(df, "FULL (merged)")
    print_class_counts(tr_df_raw, "ORIG TRAIN (raw)")
    print_class_counts(te_df_raw, "ORIG TEST (raw)")
    print_class_counts(tr_df, "ORIG TRAIN (deduped)")
    print_class_counts(te_df, "ORIG TEST (deduped)")
    print_class_counts(df_train, "TRAIN SPLIT")
    print_class_counts(df_val, "VAL SPLIT")
    print_class_counts(te_df, "TEST SPLIT")


def reduce_training_set(dedup_df, included_classes: List[str], training_size: int, defect_frac: float):
    # Use all deduplicated rows when defect_frac is 0.
    if defect_frac == 0:
        return dedup_df

    if len(included_classes) < 2:
        n = min(training_size, len(dedup_df))
        return dedup_df.sample(n=n, random_state=42)

    # defect_frac means per-defect-class fraction w.r.t. training_size
    num_defect_samples = math.floor(training_size * defect_frac)
    num_good_samples = training_size - num_defect_samples * (len(included_classes) - 1)

    if num_good_samples <= 0:
        raise ValueError(
            "Invalid training_size/defect_frac combination: non-positive good sample count. "
            f"training_size={training_size}, defect_frac={defect_frac}, classes={len(included_classes)}"
        )

    sampled_parts = []

    # sample defects
    for cls_name in included_classes[1:]:
        holder_df = dedup_df.loc[dedup_df["indication_type"] == cls_name]
        n = min(num_defect_samples, len(holder_df))
        if n < num_defect_samples:
            print(f"[WARN] class={cls_name}: requested {num_defect_samples}, available {len(holder_df)}, using {n}")
        if n > 0:
            sampled_parts.append(holder_df.sample(n=n, random_state=42))

    # sample good
    good_df = dedup_df.loc[dedup_df["indication_type"] == included_classes[0]]
    n_good = min(num_good_samples, len(good_df))
    if n_good < num_good_samples:
        print(f"[WARN] class={included_classes[0]}: requested {num_good_samples}, available {len(good_df)}, using {n_good}")
    if n_good > 0:
        sampled_parts.append(good_df.sample(n=n_good, random_state=42))

    if not sampled_parts:
        raise ValueError("No samples selected. Please adjust training_size/defect_frac.")

    reduced_df = pd.concat(sampled_parts, ignore_index=True)
    return reduced_df


In [7]:
## %% [markdown]
# ## 5. Label Map

LABEL_MAP_JSON = PROCESSED / "label_map.json"
EXPECTED_CLASSES = [
    "good",
    "color",
    "cut",
    "hole",
    "thread",
    "metal_contamination",
]


def _validate_labels(observed: List[str], label_map: Dict[str, int]) -> None:
    unknown = sorted(set(observed) - set(label_map.keys()))
    if unknown:
        raise ValueError(
            "CSV contains unknown class names (not in label_map).\n"
            f"unknown_labels={unknown}\n"
            f"label_map_keys={sorted(label_map.keys())}"
        )


def build_label_map_from_full_csv(full_csv_path: Path) -> Dict[str, int]:
    '''
    Build a stable label map from labels present in the provided CSV.
    Order always follows EXPECTED_CLASSES.
    '''
    df = pd.read_csv(full_csv_path)
    observed = set(df["indication_type"].astype(str).str.strip().unique().tolist())

    unknown = sorted(observed - set(EXPECTED_CLASSES))
    if unknown:
        raise ValueError(
            "CSV contains labels outside EXPECTED_CLASSES.\n"
            f"unknown={unknown}\n"
            f"expected={EXPECTED_CLASSES}"
        )

    ordered_present = [name for name in EXPECTED_CLASSES if name in observed]
    if not ordered_present:
        raise ValueError("No valid labels found in CSV.")

    return {name: i for i, name in enumerate(ordered_present)}


def load_or_create_label_map(PASSED_PATH: Path) -> Dict[str, int]:
    label_map = build_label_map_from_full_csv(PASSED_PATH)
    LABEL_MAP_JSON.write_text(
        json.dumps(label_map, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return label_map


def validate_split_labels(csv_path: Path, label_map: Dict[str, int]) -> None:
    df = pd.read_csv(csv_path)
    observed = df["indication_type"].astype(str).str.strip().unique().tolist()
    _validate_labels([_normalize_label(x) for x in observed], label_map)


In [8]:
# %% [markdown]
# ## 6. PyTorch Dataset & Model

class TextileDataset(Dataset):
    """Read images from full64.h5 using pointers stored in the split CSV."""

    def __init__(
        self,
        csv_path: Path,
        h5_path: Path,
        *,
        label_map: Optional[Dict[str, int]] = None,
        transform: Optional[Callable[[torch.Tensor], torch.Tensor]] = None,
        strict_labels: bool = True,
    ):
        self.df = pd.read_csv(csv_path)
        self.h5_path = str(h5_path)
        self.transform = transform

        if "abs_ptr" not in self.df.columns:
            raise ValueError(
                "CSV missing abs_ptr. Please use create_clean_split() outputs: train_split.csv/val_split.csv/test_split.csv."
            )
        if "indication_type" not in self.df.columns:
            raise ValueError("CSV missing indication_type.")

        if label_map is None:
            raise ValueError(
                "label_map is required. Call load_or_create_label_map() then pass it into TextileDataset(..., label_map=label_map)."
            )
        self.label_map = dict(label_map)

        labels = [_normalize_label(x) for x in self.df["indication_type"].tolist()]
        if strict_labels:
            _validate_labels(labels, self.label_map)
        self.df["indication_type"] = labels

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        abs_ptr = int(row["abs_ptr"])

        with h5py.File(self.h5_path, "r") as f:
            img = f["images"][abs_ptr]

        img_t = torch.from_numpy(img).float()
        if img_t.max() > 1.0:
            img_t /= 255.0

        # Ensure channel-first (1, H, W)
        if img_t.ndim == 2:
            img_t = img_t.unsqueeze(0)
        elif img_t.ndim == 3 and img_t.shape[-1] == 1:
            img_t = img_t.permute(2, 0, 1)

        if self.transform is not None:
            img_t = self.transform(img_t)

        label = self.label_map[row["indication_type"]]
        return img_t, torch.tensor(label, dtype=torch.long)


class TextileBaselineCNN(nn.Module):
    """Small CNN baseline for 64x64 grayscale images."""

    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


In [1]:
# %% [markdown]
# ## 6.5 Quick Visualization (Color Defect Samples)

def visualize_color_defects(dataset, label_map, num_samples=5):
    # Find the integer index for the 'color' class
    if "color" not in label_map:
        print("[INFO] 'color' class is not in current label_map. Skipping visualization.")
        return
    color_idx = label_map["color"]

    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    if num_samples == 1:
        axes = [axes]

    found = 0
    for img, label in dataset:
        label_val = int(label.item()) if torch.is_tensor(label) else int(label)
        if label_val == color_idx:
            # Reshape from (1, 64, 64) to (64, 64) for plotting
            axes[found].imshow(img.squeeze(), cmap="gray")
            axes[found].set_title("Defect: Color (Stain)")
            axes[found].axis("off")
            found += 1
        if found == num_samples:
            break

    if found < num_samples:
        print(f"Only found {found} color samples.")
    plt.show()


# Optional quick demo: safe to run before/after training cells.
train_split_csv = PROCESSED / "train_split.csv"
if train_split_csv.exists():
    demo_label_map = load_or_create_label_map(train_split_csv)
    demo_train_ds = TextileDataset(train_split_csv, OUT_H5, label_map=demo_label_map)
    visualize_color_defects(demo_train_ds, demo_label_map)
else:
    print("[INFO] train_split.csv not found yet. Run split-generation/training cells first, then rerun this cell.")


NameError: name 'PROCESSED' is not defined

In [9]:
# %% [markdown]
# ## 7. Training Utilities (Early Stopping)

class EarlyStopping:
    def __init__(self, patience: int = 5, verbose: bool = True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = float("inf")
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss: float, model: nn.Module) -> None:
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.best_model_state = copy.deepcopy(model.state_dict())
            self.counter = 0
            if self.verbose:
                print("Validation loss improved. Saving model weights.")
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True


def run_step(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    is_train: bool = True,
) -> Tuple[float, float]:
    model.train() if is_train else model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += float(loss.item())
            preds = outputs.argmax(dim=1)
            total += int(labels.size(0))
            correct += int((preds == labels).sum().item())

    avg_loss = total_loss / max(len(loader), 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc


In [ ]:
# %% [markdown]
# ## 8. Main (Baseline CNN)
# This section uses shared settings from TRAIN_CFG and scenario `all_training`.


In [ ]:
# Step 1: Build processed dataset and baseline split files
merge_data()
hashes = analyze_duplicates()
create_clean_split(hashes, BASELINE_SPLIT_CFG["defect_classes"], BASELINE_SPLIT_CFG["train_size"], BASELINE_SPLIT_CFG["defect_frac"])


In [ ]:
# Step 2: Build and validate label map for baseline split
label_map = load_or_create_label_map(OUT_CSV)
validate_split_labels(PROCESSED / "train_split.csv", label_map)
validate_split_labels(PROCESSED / "val_split.csv", label_map)
validate_split_labels(PROCESSED / "test_split.csv", label_map)
print("\nlabel_map:", label_map)


In [ ]:
# Step 3: Create datasets and dataloaders
train_ds = TextileDataset(PROCESSED / "train_split.csv", OUT_H5, label_map=label_map)
val_ds = TextileDataset(PROCESSED / "val_split.csv", OUT_H5, label_map=label_map)
train_loader = DataLoader(train_ds, batch_size=TRAIN_CFG["batch"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=TRAIN_CFG["batch"])


In [ ]:
# Step 4: Initialize baseline CNN training components
model = TextileBaselineCNN(num_classes=len(label_map)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=TRAIN_CFG["lr"], foreach=False)
early_stop = EarlyStopping(patience=TRAIN_CFG["patience"])


In [ ]:
# Step 5: Train baseline CNN and save best checkpoint
print(f"\nStarting training on: {device}")
for epoch in range(TRAIN_CFG["epochs"]):
    t_loss, t_acc = run_step(model, train_loader, criterion, optimizer, device, True)
    v_loss, v_acc = run_step(model, val_loader, criterion, optimizer, device, False)

    print(
        f"Epoch [{epoch + 1:02d}] | "
        f"Train: {t_acc:.2f}% (Loss: {t_loss:.4f}) | "
        f"Val: {v_acc:.2f}% (Loss: {v_loss:.4f})"
    )

    early_stop(v_loss, model)
    if early_stop.early_stop:
        print("Early stopping triggered.")
        model.load_state_dict(early_stop.best_model_state)
        break

torch.save(model.state_dict(), "best_textile_baseline.pth")
print("Training Complete.")


In [ ]:
# %% [markdown]
# ## 9. ResNet Experiments
# This section reuses `SPLIT_SCENARIOS` from Section 1.6.


In [ ]:

class TextileResNet(nn.Module):
    def __init__(self, num_classes: int):
        super(TextileResNet, self).__init__()

        # Use ResNet-18 as it is computationally efficient for 64x64 images
        # 'weights=None' means training from scratch
        self.model = models.resnet18(weights=None)

        # 1. CHANNEL ADAPTATION
        # Original ResNet-18 conv1: Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        # Change 'in_channels' from 3 (RGB) to 1 (Grayscale)
        self.model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        # 2. OUTPUT ADAPTATION
        # The original fc layer is for ImageNet (1000 classes).
        # We swap it for our specific defect count (e.g., 6 classes).
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)


In [ ]:
# Instantiate a preview ResNet (all-training scenario class count)
label_map_all = load_or_create_label_map(OUT_CSV)
model_preview = TextileResNet(len(label_map_all)).to(device)

print("ResNet-18 Model Summary for Textile Defect Detection:")
if torchinfo is not None:
    torchinfo.summary(
        model_preview,
        input_size=(TRAIN_CFG["batch"], 1, 64, 64),
        col_names=["input_size", "output_size", "num_params", "kernel_size"],
        verbose=0,
    )
else:
    print("[INFO] torchinfo not installed. Skipping model summary.")


In [ ]:
def prepare_resnet_split_and_loaders(split_cfg: Dict, train_cfg: Dict):
    create_clean_split(
        hashes,
        split_cfg["defect_classes"],
        split_cfg["train_size"],
        split_cfg["defect_frac"],
    )

    split_label_map = load_or_create_label_map(PROCESSED / "train_split.csv")
    validate_split_labels(PROCESSED / "train_split.csv", split_label_map)
    validate_split_labels(PROCESSED / "val_split.csv", split_label_map)

    split_train_ds = TextileDataset(PROCESSED / "train_split.csv", OUT_H5, label_map=split_label_map)
    split_val_ds = TextileDataset(PROCESSED / "val_split.csv", OUT_H5, label_map=split_label_map)
    split_train_loader = DataLoader(split_train_ds, batch_size=train_cfg["batch"], shuffle=True)
    split_val_loader = DataLoader(split_val_ds, batch_size=train_cfg["batch"])

    return split_label_map, split_train_ds, split_val_ds, split_train_loader, split_val_loader


In [ ]:
# Run ResNet for all 4 split scenarios
resnet_models = {}
resnet_histories = {}
resnet_label_maps = {}
resnet_loaders = {}

for scenario_name, split_cfg in SPLIT_SCENARIOS.items():
    print(f"\n=== ResNet Scenario: {scenario_name} ===")
    print(split_cfg["desc"])

    split_label_map, split_train_ds, split_val_ds, split_train_loader, split_val_loader = prepare_resnet_split_and_loaders(
        split_cfg, TRAIN_CFG
    )

    model_curr = TextileResNet(num_classes=len(split_label_map)).to(device)
    optimizer_curr = torch.optim.Adam(model_curr.parameters(), lr=TRAIN_CFG["lr"], foreach=False)
    criterion_curr = nn.CrossEntropyLoss()
    stopper_curr = EarlyStopping(patience=TRAIN_CFG["patience"], verbose=True)

    history_curr = []
    pbar = tqdm.tqdm(range(TRAIN_CFG["epochs"]), desc=f"ResNet {scenario_name}")
    for epoch in pbar:
        train_loss, train_acc = run_step(
            model_curr, split_train_loader, criterion_curr, optimizer_curr, device, True
        )
        val_loss, val_acc = run_step(
            model_curr, split_val_loader, criterion_curr, optimizer_curr, device, False
        )

        history_curr.append(
            {"Epoch": epoch + 1, "Split": "Train", "Loss": train_loss, "Accuracy": train_acc}
        )
        history_curr.append(
            {"Epoch": epoch + 1, "Split": "Val", "Loss": val_loss, "Accuracy": val_acc}
        )
        pbar.set_postfix({"val_loss": f"{val_loss:.4f}", "val_acc": f"{val_acc:.2f}"})

        stopper_curr(val_loss, model_curr)
        if stopper_curr.early_stop:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    if stopper_curr.best_model_state is not None:
        model_curr.load_state_dict(stopper_curr.best_model_state)

    resnet_models[scenario_name] = model_curr
    resnet_histories[scenario_name] = history_curr
    resnet_label_maps[scenario_name] = split_label_map
    resnet_loaders[scenario_name] = {
        "train_ds": split_train_ds,
        "val_ds": split_val_ds,
        "train_loader": split_train_loader,
        "val_loader": split_val_loader,
    }

    print(f"Completed scenario: {scenario_name}")

# Keep downstream visualization/IG on all-training scenario only
results_resnet = resnet_histories["all_training"]
model_resnet = resnet_models["all_training"]
train_label_map = resnet_label_maps["all_training"]
train_ds = resnet_loaders["all_training"]["train_ds"]
val_loader = resnet_loaders["all_training"]["val_loader"]

print("\nAll ResNet scenarios completed.")
print("results_resnet/model_resnet set to 'all_training' for plot and IG.")


In [ ]:
# --- 1. PERFORMANCE PLOT (all_training scenario only) ---
def plot_history(results_list):
    history_df = pd.DataFrame(results_list)
    metrics = ["Accuracy", "Loss"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for i, metric in enumerate(metrics):
        if sns is not None:
            sns.lineplot(data=history_df, x="Epoch", y=metric, hue="Split", ax=axes[i])
        else:
            for split in history_df["Split"].unique():
                split_df = history_df[history_df["Split"] == split]
                axes[i].plot(split_df["Epoch"], split_df[metric], label=split)
            axes[i].legend()
        axes[i].set_title(f"ResNet {metric} Over Epochs (all_training)")
    plt.tight_layout()
    plt.show()


# --- 2. THE CONFUSION MATRIX (all_training scenario only) ---
def show_confusion_matrix(model, loader, label_map, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            outputs = model(imgs.to(device))
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    class_names = list(label_map.keys())

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
    plt.title("ResNet Defect Classification - Confusion Matrix (all_training)")
    plt.show()


# Run after ResNet scenario loop
# plot_history(results_resnet)
# show_confusion_matrix(model_resnet, val_loader, train_label_map, device)


In [ ]:

def apply_integrated_gradients(model, input_tensor, target_class=None):
    """
    Computes Integrated Gradients for a specific image.
    """
    if IntegratedGradients is None:
        raise ImportError("captum is not installed. Run: pip install captum")

    model.eval()
    ig = IntegratedGradients(model)

    # Baseline (a black image of the same shape)
    baseline = torch.zeros_like(input_tensor).to(device)

    if target_class is None:
        # Get the model's predicted class
        output = model(input_tensor)
        target_class = torch.argmax(output, dim=1).item()

    # Compute attributions
    attributions, delta = ig.attribute(
        input_tensor, baseline, target=target_class, return_convergence_delta=True
    )

    # Convert to numpy for visualization [Channels, H, W] -> [H, W, Channels]
    attributions = np.transpose(attributions.squeeze().cpu().detach().numpy(), (1, 2, 0))
    img_plot = (
        np.transpose(input_tensor.squeeze().cpu().detach().numpy(), (1, 2, 0))
        if input_tensor.shape[1] > 1
        else input_tensor.squeeze().cpu().numpy()
    )

    return attributions, img_plot, target_class


def plot_ig_results(attributions, img_plot, target_class, label_map):
    if viz is None:
        raise ImportError("captum is not installed. Run: pip install captum")

    class_name = [k for k, v in label_map.items() if v == target_class][0]

    fig, ax = viz.visualize_image_attr(
        attributions,
        img_plot,
        method="blended_heat_map",
        sign="all",
        show_colorbar=True,
        title=f"IG Attribution: {class_name} (all_training)",
    )
    plt.show()
